# 02 – Подготовка данных и обучение модели

В этом ноутбуке:

* загружаем очищенный датасет `cvd_clean.csv`;
* разделяем данные на обучающую и тестовую выборки;
* строим конвейер (Pipeline) с предобработкой признаков и моделью;
* оцениваем качество модели;
* сохраняем обученный пайплайн в `models/cvd_model.pkl`.

In [ ]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

import joblib

In [ ]:
PROJECT_ROOT = Path("..").resolve()
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cvd_clean.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "cvd_model.pkl"

PROCESSED_DATA_PATH, MODEL_PATH

In [ ]:
# Загрузка очищенных данных
df = pd.read_csv(PROCESSED_DATA_PATH)
df.head()

In [ ]:
# Имя целевой колонки. Замените при необходимости.
TARGET_COLUMN = "TenYearCHD"

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Целевая колонка {TARGET_COLUMN!r} не найдена. "
        "Проверьте имя столбца и измените переменную TARGET_COLUMN."
    )

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X.head()

In [ ]:
# Определяем типы признаков
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "bool", "category"]).columns.tolist()

numeric_features, categorical_features

In [ ]:
# Разделение на train / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

In [ ]:
# Конвейер предобработки
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [ ]:
# Модель: логистическая регрессия
log_reg = LogisticRegression(max_iter=1000)

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", log_reg),
])

clf

In [ ]:
# Обучение модели
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_proba)

print(f"Accuracy: {acc:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall: {rec:.3f}")
print(f"ROC-AUC: {roc:.3f}")

In [ ]:
# Сохранение обученного пайплайна
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(clf, MODEL_PATH)

MODEL_PATH